# 🛒 Online Shoppers Purchasing Intention: Classification Benchmark
**Authors:** Ujjwal Singh (262IS036) & Aryan (262IS007)
**Course:** CS801 Computing Lab, M.Tech, NITK Surathkal

### Objective:
Classify e-commerce shopping sessions into purchase (`Revenue = 1`) or browsing only (`Revenue = 0`) across 12,330 sessions using pure classical (non-tree) classification models:
1. Logistic Regression (Baseline)
2. Ridge-penalized Logistic (L2 Penalty)
3. Lasso-penalized Logistic (L1 Penalty)
4. K-Nearest Neighbors Classifier (KNN)
5. Gaussian Naive Bayes (Probabilistic)
6. Support Vector Classifier (Linear SVC)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

df = pd.read_csv("../../datasets/online_shoppers.csv")
df['Revenue'] = df['Revenue'].astype(int)
df['Weekend'] = df['Weekend'].astype(int)
print("Dataset Shape:", df.shape)
df.head()


## 1. Stratified 80/20 Train-Test Split

In [ ]:
X = df.drop(columns=['Revenue'])
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Training sessions:", len(X_train), "| Testing sessions:", len(X_test))


## 2. Preprocessing Architecture
Numerical features are standardized, while categorical visitor features are one-hot encoded without sparsity for compatibility with Gaussian Naive Bayes and linear classifiers.


In [ ]:
num_features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
cat_features = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)


## 3. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression (Baseline)': LogisticRegression(max_iter=1000, random_state=42),
    'Ridge Logistic (L2, C=0.1)': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Lasso Logistic (L1, C=0.5)': LogisticRegression(penalty='l1', solver='liblinear', C=0.5, max_iter=1000, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Gaussian Naive Bayes': GaussianNB(),
    'Support Vector Machine (Linear SVC)': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=3000))
}

results = []
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else None
    auc = round(float(roc_auc_score(y_test, probs)), 4) if probs is not None else 0.0
    
    results.append({
        'Model': name,
        'Accuracy (%)': round(float(accuracy_score(y_test, preds) * 100), 2),
        'Precision (%)': round(float(precision_score(y_test, preds, zero_division=0) * 100), 2),
        'Recall (%)': round(float(recall_score(y_test, preds, zero_division=0) * 100), 2),
        'F1-Score (%)': round(float(f1_score(y_test, preds, zero_division=0) * 100), 2),
        'ROC-AUC': auc
    })

scorecard = pd.DataFrame(results).sort_values('F1-Score (%)', ascending=False)
scorecard
